# Evaluación de generadores 3D (sección 5.4)

**Objetivo:** elegir el generador con criterios medibles (calidad, tiempo y costo), como pide la propuesta, antes de integrarlo al backend.

| Opción | Flujo | Qué se mide acá |
|---|---|---|
| **TripoSR** | texto → imagen (SDXL-Turbo) → TripoSR → `.glb` | tiempo de cada paso, geometría, vista previa |
| **Shap-E** | texto → `.glb` | tiempo, geometría, vista previa |
| API comercial | texto → `.glb` (servicio externo) | no se evalúa acá: necesita una clave y tiene costo por modelo |

## Cómo correrlo
1. En Colab: **Entorno de ejecución → Cambiar tipo de entorno → GPU (T4)**.
2. **Entorno de ejecución → Ejecutar todas**. Tarda entre 20 y 30 minutos; la mayor parte es instalar y descargar modelos.
3. Al final se descarga `resultados_generacion3d.zip`. Hay que completar la columna de calidad del CSV (sección 4) y guardar todo en `eval/generacion3d/resultados/` del repositorio.

Si una celda de instalación falla por versiones de paquetes, **Entorno de ejecución → Reiniciar sesión** y volver a ejecutar desde el principio.

In [ ]:
import subprocess
import torch

print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout)
assert torch.cuda.is_available(), "No hay GPU: Entorno de ejecución → Cambiar tipo de entorno → GPU"

## Objetos de prueba

Tres piezas que ya están en el catálogo (para comparar con los modelos hechos a mano) y dos que **no** están, que es el caso real en el que el producto tendría que generar.

Los prompts en inglés siguen el **prompt B** de la sección 14.1 (`backend/prompts/gen3d_prompt.txt`): un solo objeto centrado, fondo blanco liso, vista tres cuartos, sin texto ni personas, estilo realista y materiales claros. En el producto ese texto lo escribe el LLM; acá se dejan escritos para que la prueba sea repetible.
Shap-E funciona mejor con frases cortas, por eso cada objeto tiene además un prompt corto.

In [ ]:
ESTILO = ("centered, plain white background, three-quarter view, no text, no people, "
          "realistic style, clear materials")

OBJETOS = [
    {"id": "valvula", "descripcion": "válvula de compuerta con volante", "en_catalogo": True,
     "prompt": f"a single industrial steel gate valve with a red handwheel and bolted flanges, {ESTILO}",
     "prompt_corto": "an industrial steel gate valve with a red handwheel"},
    {"id": "engranaje", "descripcion": "engranaje recto", "en_catalogo": True,
     "prompt": f"a single steel spur gear with straight teeth and a central hole, {ESTILO}",
     "prompt_corto": "a steel spur gear"},
    {"id": "rodamiento", "descripcion": "rodamiento de bolas", "en_catalogo": True,
     "prompt": f"a single steel ball bearing with visible balls between the inner and outer rings, {ESTILO}",
     "prompt_corto": "a steel ball bearing"},
    {"id": "llave_inglesa", "descripcion": "llave inglesa", "en_catalogo": False,
     "prompt": f"a single chrome adjustable wrench, {ESTILO}",
     "prompt_corto": "a chrome adjustable wrench"},
    {"id": "bomba_agua", "descripcion": "bomba de agua centrífuga", "en_catalogo": False,
     "prompt": f"a single blue industrial centrifugal water pump with an electric motor on a metal base, {ESTILO}",
     "prompt_corto": "a blue industrial centrifugal water pump"},
]

from pathlib import Path
SALIDA = Path("/content/resultados_generacion3d")
SALIDA.mkdir(exist_ok=True)
resultados = []   # una fila por generador y objeto
cargas = {}       # tiempo de carga de cada modelo (arranque en frío)

## 1. Texto → imagen (SDXL-Turbo)

Primer paso del flujo de TripoSR, que parte de una imagen. SDXL-Turbo genera una imagen en un solo paso de inferencia.

In [ ]:
!pip install -q diffusers transformers accelerate trimesh fast-simplification

In [ ]:
import gc
import time

import matplotlib.pyplot as plt
from diffusers import AutoPipelineForText2Image

t0 = time.perf_counter()
sdxl = AutoPipelineForText2Image.from_pretrained("stabilityai/sdxl-turbo", torch_dtype=torch.float16,
                                                 variant="fp16").to("cuda")
cargas["SDXL-Turbo"] = time.perf_counter() - t0

tiempos_imagen = {}
for obj in OBJETOS:
    t0 = time.perf_counter()
    imagen = sdxl(prompt=obj["prompt"], num_inference_steps=1, guidance_scale=0.0,
                  generator=torch.Generator("cuda").manual_seed(0)).images[0]
    tiempos_imagen[obj["id"]] = time.perf_counter() - t0
    imagen.save(SALIDA / f"imagen_{obj['id']}.png")

fig, ejes = plt.subplots(1, len(OBJETOS), figsize=(4 * len(OBJETOS), 4))
for eje, obj in zip(ejes, OBJETOS):
    eje.imshow(plt.imread(SALIDA / f"imagen_{obj['id']}.png"))
    eje.set_title(f"{obj['descripcion']}\n{tiempos_imagen[obj['id']]:.1f} s")
    eje.axis("off")
plt.show()

del sdxl
gc.collect()
torch.cuda.empty_cache()

## 2. TripoSR (imagen → 3D)

La instalación compila `torchmcubes` (marching cubes en GPU): puede tardar unos minutos.

In [ ]:
%cd /content
!git clone -q https://github.com/VAST-AI-Research/TripoSR.git
!pip install -q omegaconf einops rembg onnxruntime xatlas
!pip install -q git+https://github.com/tatsy/torchmcubes.git

In [ ]:
import inspect
import sys

import numpy as np
import rembg
from PIL import Image

sys.path.insert(0, "/content/TripoSR")
from tsr.system import TSR
from tsr.utils import remove_background, resize_foreground

t0 = time.perf_counter()
tsr = TSR.from_pretrained("stabilityai/TripoSR", config_name="config.yaml", weight_name="model.ckpt")
tsr.renderer.set_chunk_size(8192)
tsr.to("cuda")
cargas["TripoSR"] = time.perf_counter() - t0
sesion_rembg = rembg.new_session()


def preparar_imagen(imagen):
    # Mismo preprocesamiento que run.py de TripoSR: sacar el fondo y centrar el objeto
    imagen = resize_foreground(remove_background(imagen, sesion_rembg), 0.85)
    arr = np.array(imagen).astype(np.float32) / 255.0
    arr = arr[:, :, :3] * arr[:, :, 3:4] + (1 - arr[:, :, 3:4]) * 0.5
    return Image.fromarray((arr * 255.0).astype(np.uint8))


# La firma de extract_mesh cambió entre versiones de TripoSR
opciones_malla = {"resolution": 256}
if "has_vertex_color" in inspect.signature(tsr.extract_mesh).parameters:
    opciones_malla["has_vertex_color"] = True

for obj in OBJETOS:
    t0 = time.perf_counter()
    imagen = preparar_imagen(Image.open(SALIDA / f"imagen_{obj['id']}.png"))
    with torch.no_grad():
        codigos = tsr([imagen], device="cuda")
    malla = tsr.extract_mesh(codigos, **opciones_malla)[0]
    segundos_3d = time.perf_counter() - t0
    archivo = f"triposr_{obj['id']}.glb"
    malla.export(SALIDA / archivo)
    resultados.append({"generador": "TripoSR", "objeto": obj["id"], "en_catalogo": obj["en_catalogo"],
                       "seg_imagen": round(tiempos_imagen[obj["id"]], 2), "seg_3d": round(segundos_3d, 2),
                       "seg_total": round(tiempos_imagen[obj["id"]] + segundos_3d, 2), "archivo": archivo})
    print(f"{obj['id']}: {segundos_3d:.1f} s")

del tsr
gc.collect()
torch.cuda.empty_cache()

## 3. Shap-E (texto → 3D)

In [ ]:
import numpy as np
import trimesh
from diffusers import ShapEPipeline
from diffusers.utils import export_to_ply

t0 = time.perf_counter()
shap_e = ShapEPipeline.from_pretrained("openai/shap-e", torch_dtype=torch.float16, variant="fp16").to("cuda")
cargas["Shap-E"] = time.perf_counter() - t0

for obj in OBJETOS:
    t0 = time.perf_counter()
    salida = shap_e(obj["prompt_corto"], guidance_scale=15.0, num_inference_steps=64, frame_size=256,
                    output_type="mesh", generator=torch.Generator("cuda").manual_seed(0)).images[0]
    ply = export_to_ply(salida, str(SALIDA / f"shape_{obj['id']}.ply"))
    malla = trimesh.load(ply)
    # Shap-E usa Z hacia arriba; el visor (glTF) usa Y hacia arriba
    malla.apply_transform(trimesh.transformations.rotation_matrix(-np.pi / 2, [1, 0, 0]))
    segundos = time.perf_counter() - t0
    archivo = f"shape_{obj['id']}.glb"
    malla.export(SALIDA / archivo, file_type="glb")
    resultados.append({"generador": "Shap-E", "objeto": obj["id"], "en_catalogo": obj["en_catalogo"],
                       "seg_imagen": 0.0, "seg_3d": round(segundos, 2), "seg_total": round(segundos, 2),
                       "archivo": archivo})
    print(f"{obj['id']}: {segundos:.1f} s")

del shap_e
gc.collect()
torch.cuda.empty_cache()

## 4. Comparación

Métricas automáticas por modelo: tiempo, cantidad de vértices y caras, tamaño del `.glb` y si la malla es **estanca** (cerrada, sin agujeros).
La **calidad** la puntúan personas: 2 o 3 integrantes, de 1 a 5, en las columnas `calidad_forma` (¿se reconoce el objeto?) y `calidad_detalle` (¿tiene las partes importantes, sin artefactos?). Después se promedia.

In [ ]:
import numpy as np
import pandas as pd
import trimesh


def cargar_malla(ruta):
    return trimesh.load(ruta, force="mesh")


filas = []
for r in resultados:
    malla = cargar_malla(SALIDA / r["archivo"])
    filas.append({**r, "vertices": len(malla.vertices), "caras": len(malla.faces),
                  "kb": round((SALIDA / r["archivo"]).stat().st_size / 1024),
                  "estanca": bool(malla.is_watertight)})
tabla = pd.DataFrame(filas)
for columna in ("calidad_forma", "calidad_detalle", "notas"):
    tabla[columna] = ""
tabla.to_csv(SALIDA / "resultados.csv", index=False)

print("Carga de cada modelo (arranque en frío, segundos):", {k: round(v, 1) for k, v in cargas.items()})
display(tabla.drop(columns=["archivo", "calidad_forma", "calidad_detalle", "notas"]))
display(tabla.groupby("generador")[["seg_total", "vertices", "kb"]].mean().round(1))

In [ ]:
import matplotlib.pyplot as plt


def vista_previa(eje, ruta, titulo, max_caras=20000):
    # Render simple con matplotlib: funciona en Colab sin pantalla
    malla = cargar_malla(ruta)
    if len(malla.faces) > max_caras:
        # Solo las mallas muy densas (TripoSR genera más de 100.000 caras). Simplificar conserva
        # la forma; tomar caras al azar dejaría la malla llena de agujeros.
        malla.merge_vertices()
        try:
            malla = malla.simplify_quadric_decimation(face_count=max_caras)
        except Exception as e:
            print(f"No se pudo simplificar {ruta.name} ({e}); se dibuja completa")
    caras = malla.faces
    v = malla.vertices - malla.centroid
    eje.plot_trisurf(v[:, 0], v[:, 2], v[:, 1], triangles=caras, color="#7fb8d0",
                     linewidth=0, antialiased=False, shade=True)
    lim = np.abs(v).max()
    eje.set_xlim(-lim, lim)
    eje.set_ylim(-lim, lim)
    eje.set_zlim(-lim, lim)
    eje.view_init(elev=20, azim=-60)
    eje.set_title(titulo, fontsize=9)
    eje.axis("off")


generadores = list(dict.fromkeys(r["generador"] for r in resultados))
fig = plt.figure(figsize=(4 * (len(generadores) + 1), 4 * len(OBJETOS)))
for i, obj in enumerate(OBJETOS):
    columnas = len(generadores) + 1
    eje = fig.add_subplot(len(OBJETOS), columnas, i * columnas + 1)
    ruta_imagen = SALIDA / f"imagen_{obj['id']}.png"
    if ruta_imagen.exists():
        eje.imshow(plt.imread(ruta_imagen))
    eje.set_title(f"{obj['descripcion']}\n(imagen de entrada)", fontsize=9)
    eje.axis("off")
    for j, generador in enumerate(generadores):
        fila = next((r for r in resultados if r["generador"] == generador and r["objeto"] == obj["id"]), None)
        eje = fig.add_subplot(len(OBJETOS), columnas, i * columnas + j + 2, projection="3d")
        if fila:
            vista_previa(eje, SALIDA / fila["archivo"], f"{generador} · {fila['seg_total']:.0f} s")
plt.tight_layout()
plt.savefig(SALIDA / "comparacion.png", dpi=110)
plt.show()

Las vistas previas son aproximadas. Para juzgar la calidad conviene abrir los `.glb` en el visor del proyecto: copiarlos a `frontend/models/` y cargarlos, o arrastrarlos a <https://gltf-viewer.donmccurdy.com/>.

In [ ]:
import shutil

shutil.make_archive("/content/resultados_generacion3d", "zip", SALIDA)
try:
    from google.colab import files
    files.download("/content/resultados_generacion3d.zip")
except ImportError:
    print("Archivo listo: /content/resultados_generacion3d.zip")

## 5. Decisión (completar con los resultados)

| Criterio | Peso | TripoSR | Shap-E | API comercial |
|---|---|---|---|---|
| Calidad (promedio de forma y detalle) | 40 % | | | |
| Tiempo por modelo | 25 % | | | |
| Costo y dependencia (GPU, internet, pago) | 20 % | | | |
| Facilidad de integración al backend | 15 % | | | |

Preguntas para la defensa: ¿la calidad alcanza para mostrar un producto a un cliente? ¿El tiempo es aceptable con un indicador de carga (sección 15: "reportar; mostrar indicador de carga")?
Si ninguna opción alcanza, la sección 10.2 permite recortar la generación en vivo y mostrar modelos generados de antemano (riesgo 1, plan B).